In [ ]:
import pandas as pd 
import random 
import math 
import numpy as np 
import matplotlib.pyplot as plt 

In [ ]:
df = pd.read_csv("/kaggle/input/job-description-dataset/job_descriptions.csv")

In [ ]:
df.shape

### Data formating

In [ ]:
random.seed(10)

rand_idx = set()
while len(rand_idx) < 30000:
    rand_idx.add(random.randint(0, len(df)-1))
    
# Select the rows from df
random_rows = df.iloc[list(rand_idx)]

In [ ]:
df_s = pd.DataFrame(random_rows)
df_s.shape

In [ ]:
df_s.head(5)

In [ ]:
df_s.reset_index(drop=True).head(3)

In [ ]:
df_s.head(3)

In [ ]:
df_s = df_s.reset_index(drop = True)
df_s.head(3)

### Understanding our data

In [ ]:
df_s.info()

In [ ]:
df_s = df_s.drop(['latitude', 'longitude', 'Contact Person', 'Contact', 'Company Profile', 'Job Description', 'Benefits', 'Job Title'], axis = 1)

In [ ]:
df_s.head(3)

Three columns to reformat: Experience, Salary Range and Job Posting Date

In [ ]:
df_s[['Salary min', 'Salary max']] = df_s['Salary Range'].str.split('-', expand = True)

In [ ]:
df_s.head(3)

In [ ]:
df_s = df_s.drop(['Salary Range'], axis = 1)
df_s.head(3)

In [ ]:
df_s['Salary min'] = df_s['Salary min'].str.replace('$', '').str.rstrip('K').astype(float)*1000
df_s['Salary max'] = df_s['Salary max'].str.replace('$', '').str.rstrip('K').astype(float)*1000
df_s.head(3)

In [ ]:
df_s[['Experience min', 'Experience max']] = df_s['Experience'].str.split(' to ', expand=True)

# Convert strings to integers
df_s['Experience min'] = df_s['Experience min'].str.replace(' Years', '').astype(int)
df_s['Experience max'] = df_s['Experience max'].str.replace(' Years', '').astype(int)
df_s.head(3)

In [ ]:
df_s = df_s.drop(['Experience'], axis = 1)
df_s.head(3)

In [ ]:
df_s['Job Posting Date']

In [ ]:
pd.to_datetime(df_s['Job Posting Date'])

In [ ]:
df_s['Job Posting Date'] = pd.to_datetime(df_s['Job Posting Date'])
df_s.head(3)

In [ ]:
df_s.info()

## Data cleaning

- Handle duplicates
    - Drop and leave one
    - Merge, mean or median
- Find missing entries 
- Scaling your data
    - min-max scaling

------
**Data normalization**  

Normalization is necessary when attributes have different scales, as large-scale attributes can dominate smaller-scale ones, leading to biased data models. If attributes vary significantly in scale, the model's performance may suffer. Normalization brings all attributes to a common scale, ensuring that no single feature disproportionately influences the analysis, ultimately improving the effectiveness of data mining models.

        > More info: https://developers.google.com/machine-learning/data-prep/transform/normalization

In [ ]:
df_s[df_s.duplicated(subset=['Job Id'])] # Do not keep any duplicates

In [ ]:
df_s[df_s.isnull().any(axis=1)]

In [ ]:
df_s.describe()

In [ ]:
df_s.head()

In [ ]:
def min_max_scale(column):
    min_val = column.min()
    max_val = column.max()
    scaled_column = (column - min_val) / (max_val - min_val)
    return scaled_column

In [ ]:
for col_name in ['Salary min', 'Salary max', 'Experience min', 'Experience max', 'Company Size']:
    df_s[col_name + '_sc'] = min_max_scale(df_s[col_name])
    df_s = df_s.drop([col_name], axis = 1)
    
df_s.head(3)

## Ask questions to the data

**Highest income**: Which countries and jobs have the highest income?

In [ ]:
df_s[['Country', 'Role', 'Salary max_sc']].sort_values(by=['Salary max_sc'], ascending=False).head(5)

**Job opportunities:** Which country has the most job opportunities? 

In [ ]:
df_s['Country'].value_counts().to_frame()

**Qualifications**: What are the most commonly required qualifications for the listed jobs?

In [ ]:
print('Unique qualifications:')
print(set(df_s.Qualifications.tolist()))

In [ ]:
qualifications_counts = df_s['Qualifications'].value_counts()
print('Most common qualifications:')
qualifications_counts.head(5)

**Job Posting Trends**: How are job postings distributed over time? Are there any seasonal patterns or fluctuations in job postings?

In [ ]:
df_s['Month'] = df_s['Job Posting Date'].dt.month
df_s['Year'] = df_s['Job Posting Date'].dt.year

In [ ]:
posting_trends = df_s.groupby(['Year', 'Month']).size()

In [ ]:
posting_trends.to_frame().transpose()